# Interactive Demo

This notebook is modified from https://github.com/facebookresearch/sam2/blob/main/notebooks/video_predictor_example.ipynb.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import os
os.chdir("..")

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from imageio.v3 import imread, imwrite
from IPython.display import Video
from sam2.sam2_video_predictor import SAM2VideoPredictor
from tqdm import tqdm
from vis_utils.utils import dilute_image, put_texts

from prep_splart_real import main as prep_data

In [ ]:
def show_mask(mask, ax, obj_id=None, random_color=False):
    if random_color:
        color = np.array((*np.random.random(3), 0.6))
    else:
        cmap = plt.get_cmap("tab10")
        cmap_idx = 0 if obj_id is None else obj_id
        color = np.array((*cmap(cmap_idx)[:3], 0.6))
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)


def show_points(coords, labels, ax, marker_size=200):
    pos_points = coords[labels == 1]
    neg_points = coords[labels == 0]
    ax.scatter(
        pos_points[:, 0], pos_points[:, 1], color="green", marker="*", s=marker_size, edgecolor="white", linewidth=1.25
    )
    ax.scatter(
        neg_points[:, 0], neg_points[:, 1], color="red", marker="*", s=marker_size, edgecolor="white", linewidth=1.25
    )


def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor="green", facecolor=(0, 0, 0, 0), lw=2))


assert torch.cuda.is_available()
# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
# turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

predictor = SAM2VideoPredictor.from_pretrained("facebook/sam2.1-hiera-large")

## Extract images

In [ ]:
dataset = "splart-real"
scene = "monitor_3"
dataset_dir = Path(f"datasets/splart/{dataset}/{scene}")
prep_data(dataset_dir, n_frames=128, extract_images=True)

## Foreground segmentation

### Starting articulation state

Let's begin with the starting articulation state.

Load the frames and take a look at the first one.

In [ ]:
articulation_state = 0
video_dir = dataset_dir / f"images/{articulation_state}"

# initialize the inference state
inference_state = predictor.init_state(str(video_dir))

# scan all the JPEG frame names in this directory
frame_paths = sorted([p for p in video_dir.iterdir() if p.suffix == ".jpg"])
imgs = [imread(p) for p in frame_paths]

# take a look at the first frame
if "fig0" in locals():
    plt.close(fig0)
fig0 = plt.figure()
frame_idx = 0
plt.title(f"frame {frame_idx}")
plt.axis(False)
plt.imshow(imgs[frame_idx])

#### Segment the foreground using the box+points prompt
<a id='prompting_0'></a>

Tip: when setting the points/box coords, you may mouse over the image from the previous step to get an idea.

In [ ]:
ann_frame_ids = (0,)  # ids of the frames to annotate

boxes = (
    np.array(
        (279, 463, 731, 921)
    ),  # (x_min, y_min, x_max, y_max); each annotation frame only supports up to one box prompt
)
points = (None,)  # ((xi, yi),); each annotation frame supports multiple point prompts
# for labels, `1` means positive click and `0` means negative click
labels = (None,)  # (label_i,)

out_mask_logits = []
predictor.reset_state(inference_state)
for i, ann_frame_id in enumerate(ann_frame_ids):
    # send the box input along with the click together into `add_new_points_or_box`
    out_mask_logits.append(
        predictor.add_new_points_or_box(
            inference_state, ann_frame_id, 0, points=points[i], labels=labels[i], box=boxes[i]
        )[2]
    )

if "figs0" in locals():
    for fig in figs0:
        plt.close(fig)
figs0 = []
for i, ann_frame_id in enumerate(ann_frame_ids):
    # show the results on the current (interacted) frame
    figs0.append(plt.figure())
    plt.title(f"frame {ann_frame_id}")
    plt.axis(False)
    plt.imshow(imgs[ann_frame_id])
    box = boxes[i]
    pts = points[i]
    if box is not None:
        show_box(box, figs0[-1].gca())
    if pts is not None:
        show_points(pts, labels[i], figs0[-1].gca())
    show_mask((out_mask_logits[i] > 0.0).cpu().numpy(), figs0[-1].gca(), obj_id=0)

#### Propagate through the video

In [ ]:
# run propagation throughout the video and collect the results in a dict
video_segments = {}  # video_segments contains the per-frame segmentation results
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy() for i, out_obj_id in enumerate(out_obj_ids)
    }
imgs_vis = []
for i, img in enumerate(tqdm(imgs)):
    img_vis = img.copy()
    dilute_image(img_vis, ~video_segments[i][0][0], alpha=200)
    put_texts(img_vis, f"frame {i:04d}", color=(255, 128, 0))
    imgs_vis.append(img_vis)
output_path = Path(f"outputs/segmented/{scene}/{articulation_state}.mp4")
output_path.parent.mkdir(exist_ok=True, parents=True)
imwrite(output_path, imgs_vis, fps=10, quality=10)
Video(output_path)

#### Export the masks

Before proceeding, carefully examine the visualization from the last step. If there are any frames with substantial segmentation error, go back to the [Segment the foreground using the box+points prompt](#prompting_0) step and add prompts for that frame. Repeat the process until all frames look good. Typically only 1~3 frame annotations will be enough.

In [ ]:
foregrounds_dir = dataset_dir / f"foregrounds/{articulation_state}"
foregrounds_dir.mkdir(exist_ok=True, parents=True)
backgrounds_dir = dataset_dir / f"backgrounds/{articulation_state}"
backgrounds_dir.mkdir(exist_ok=True, parents=True)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (16, 16))
for i, video_segment in tqdm(video_segments.items()):
    mask = video_segment[0][0, ..., None].astype(np.uint8)
    foreground = np.concatenate((imgs[i] * mask, mask * 255), axis=-1)
    imwrite(foregrounds_dir / f"{i:04d}.png", foreground)
    mask = 1 - cv2.dilate(mask, kernel)[..., None]
    background = np.concatenate((imgs[i] * mask, mask * 255), axis=-1)
    imwrite(backgrounds_dir / f"{i:04d}.png", background)

### Ending articulation state

We repeat the same process for the ending articulation state.

Load the frames and take a look at the first one.

In [ ]:
articulation_state = 1
video_dir = dataset_dir / f"images/{articulation_state}"

# initialize the inference state
inference_state = predictor.init_state(str(video_dir))

# scan all the JPEG frame names in this directory
frame_paths = sorted([p for p in video_dir.iterdir() if p.suffix == ".jpg"])
imgs = [imread(p) for p in frame_paths]

# take a look at the first frame
if "fig1" in locals():
    plt.close(fig1)
fig1 = plt.figure()
frame_idx = 0
plt.title(f"frame {frame_idx}")
plt.axis(False)
plt.imshow(imgs[frame_idx])

#### Segment the foreground using the box+points prompt
<a id='prompting_1'></a>

Tip: when setting the points/box coords, you may mouse over the image from the previous step to get an idea.

In [ ]:
ann_frame_ids = (0, 9, 55)  # the frame ids we interact with

boxes = (
    np.array(
        (331, 505, 757, 962)
    ),  # (x_min, y_min, x_max, y_max); each annotation frame only supports up to one box prompt
    np.array((456, 427, 819, 1061)),
    np.array((217, 645, 850, 1102)),
)
points = (
    None,  # ((xi, yi),); each annotation frame supports multiple point prompts
    np.array(((726, 593),)),
    np.array(((539, 687),)),
)
# for labels, `1` means positive click and `0` means negative click
labels = (None, np.array((1,)), np.array((1,)))  # (label_i,)

out_mask_logits = []
predictor.reset_state(inference_state)
for i, ann_frame_id in enumerate(ann_frame_ids):
    # send the box input along with the click together into `add_new_points_or_box`
    out_mask_logits.append(
        predictor.add_new_points_or_box(
            inference_state, ann_frame_id, 0, points=points[i], labels=labels[i], box=boxes[i]
        )[2]
    )

if "figs1" in locals():
    for fig in figs1:
        plt.close(fig)
figs1 = []
for i, ann_frame_id in enumerate(ann_frame_ids):
    # show the results on the current (interacted) frame
    figs1.append(plt.figure())
    plt.title(f"frame {ann_frame_id}")
    plt.axis(False)
    plt.imshow(imgs[ann_frame_id])
    box = boxes[i]
    pts = points[i]
    if box is not None:
        show_box(box, figs1[-1].gca())
    if pts is not None:
        show_points(pts, labels[i], figs1[-1].gca())
    show_mask((out_mask_logits[i] > 0.0).cpu().numpy(), figs1[-1].gca(), obj_id=0)

#### Propagate through the video

In [ ]:
# run propagation throughout the video and collect the results in a dict
video_segments = {}  # video_segments contains the per-frame segmentation results
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        out_obj_id: (out_mask_logits[i] > 0.0).cpu().numpy() for i, out_obj_id in enumerate(out_obj_ids)
    }
imgs_vis = []
for i, img in enumerate(tqdm(imgs)):
    img_vis = img.copy()
    dilute_image(img_vis, ~video_segments[i][0][0], alpha=200)
    put_texts(img_vis, f"frame {i:04d}", color=(255, 128, 0))
    imgs_vis.append(img_vis)
output_path = Path(f"outputs/segmented/{scene}/{articulation_state}.mp4")
output_path.parent.mkdir(exist_ok=True, parents=True)
imwrite(output_path, imgs_vis, fps=10, quality=10)
Video(output_path)

#### Export the masks

Before proceeding, carefully examine the visualization from the last step. If there are any frames with substantial segmentation error, go back to the [Segment the foreground using the box+points prompt](#prompting_1) step and add prompts for that frame. Repeat the process until all frames look good. Typically only 1~3 frame annotations will be enough.

In [ ]:
foregrounds_dir = dataset_dir / f"foregrounds/{articulation_state}"
foregrounds_dir.mkdir(exist_ok=True, parents=True)
backgrounds_dir = dataset_dir / f"backgrounds/{articulation_state}"
backgrounds_dir.mkdir(exist_ok=True, parents=True)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (16, 16))
for i, video_segment in tqdm(video_segments.items()):
    mask = video_segment[0][0, ..., None].astype(np.uint8)
    foreground = np.concatenate((imgs[i] * mask, mask * 255), axis=-1)
    imwrite(foregrounds_dir / f"{i:04d}.png", foreground)
    mask = 1 - cv2.dilate(mask, kernel)[..., None]
    background = np.concatenate((imgs[i] * mask, mask * 255), axis=-1)
    imwrite(backgrounds_dir / f"{i:04d}.png", background)

### Calibrate cameras with background images

To avoid out-of-memory issues, it's highly recommended to restart the kernel now.

In [ ]:
# IMPORTANT: This cell should only be executed following a kernel restart.

%load_ext autoreload
%autoreload 2

import os
os.chdir("..")

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from IPython.display import Video
from nerfstudio.cameras.cameras import Cameras
from nerfstudio.process_data.process_data_utils import list_images
from nerfstudio.utils.scripts import run_command
from vis_utils.utils import gen_spherical_spiral_poses, get_cur_timestamp

from prep_splart_real import main as prep_data
from splart.articulation_params import ArticulationType
from splart_renderer import SplartRenderer

The calibration may take a while (5~30 minutes depending on the computational power).

In [ ]:
dataset = "splart-real"
scene = "monitor_3"
dataset_dir = Path(f"datasets/splart/{dataset}/{scene}")
prep_data(dataset_dir, run_sfm=True, vis=True)

If the calibration looks good, we export to the required format by SplArt. Otherwise, either improve the segmentation quality or start over from the data collection.

Tips:
1. Move the camera steadily to reduce motion blur.
2. Keep the camera at an appropriate distance from the articulated object: close enough to see the details, while far enough to see the background, which is used for camera calibration.
3. Decorate the background to make the camera calibration easier.

IMPORTANT: Make sure to set the articulation type below.

In [ ]:
img_paths = list_images(dataset_dir / "foregrounds")
calib_img_paths = list_images(dataset_dir / "sfm/images")
with (dataset_dir / "sfm/transforms.json").open() as f:
    meta = json.load(f)
frames = {frame["file_path"]: frame for frame in meta["frames"]}
del meta["frames"]
del meta["ply_file_path"]
# HERE: set the articulation type. Choose between REVOLUTE and PRISMATIC.
meta["articulation"] = {"type": ArticulationType.PRISMATIC}
data = {"frames": []}
for img_path, calib_img_path in zip(img_paths, calib_img_paths):
    img_name = img_path.relative_to(dataset_dir)
    calib_img_name = calib_img_path.relative_to(dataset_dir / "sfm")
    frame = frames[str(calib_img_name)]
    frame["file_path"] = str(img_name)
    frame["state"] = int(img_name.parent.name)
    data["frames"].append(frame)
meta.update(data)
with (dataset_dir / "transforms.json").open("w") as f:
    json.dump(meta, f, indent=2)

### Reconstruct the articulated object

In [ ]:
timestamp = get_cur_timestamp()
run_command(
    f"ns-train splart \
    --output-dir model_ckpts/{dataset} \
    --experiment-name {scene}/{timestamp} \
    --vis wandb \
    --data {dataset_dir} \
    --max-num-iterations 25000 \
    --pipeline.model.num-random 999999 \
    --pipeline.model.random-scale 1.3",
    verbose=True,
)

### Visualize the reconstruction

We can render the reconstruction at novel viewpoints and articulation states.

In [ ]:
model_ckpt_dir = max((Path(f"model_ckpts/{dataset}/{scene}/{timestamp}/splart")).iterdir()) / "nerfstudio_models"
splart_renderer = SplartRenderer(model_ckpt_dir, data_dir=dataset_dir)
output_dir = Path(f"outputs/renders/{dataset}/{scene}")
fps = 20
duration = 20
n = fps * duration
cams = np.array(gen_spherical_spiral_poses(1, 0, np.pi / 3, -0.25, 1.25, n=n))
cams = Cameras(torch.from_numpy(cams), 800.0, 800.0, 500.0, 500.0)
articulation_states = (np.sin(np.arange(n) / (n - 1) * np.pi * 15 - np.pi / 2) + 1) / 2
splart_renderer.render_views(
    cams, output_dir, name=timestamp, articulation_states=articulation_states, pose_type="cam2ns", animate=fps
)
Video(output_dir / timestamp / "color.mp4")